In [ ]:
import numpy as np
import cv2 as cv
import open3d as o3d
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
imgL = cv.imread('/Users/luryand/Documents/VC/img/task04/side_l1.jpg')
imgR = cv.imread('/Users/luryand/Documents/VC/img/task04/side_r1.jpg')
imgL_gray = cv.cvtColor(imgL, cv.COLOR_BGR2GRAY)
imgR_gray = cv.cvtColor(imgR, cv.COLOR_BGR2GRAY)

stereo = cv.StereoSGBM_create(numDisparities=64,blockSize=9, 
                              P1=8*3*9**2, P2=32*3*9**2, 
                              disp12MaxDiff=1, uniquenessRatio=10, 
                              speckleWindowSize=0, speckleRange=0, 
                              preFilterCap=63, minDisparity=0)
disparity = stereo.compute(imgL_gray, imgR_gray).astype(np.int16)

disp_norm = cv.normalize(disparity, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)
disp_blur = cv.medianBlur(disp_norm, 21)

In [ ]:
# Matrizes de calibração (exemplo: use sua matriz calibrada)
calib_matrix_P2 = np.array([
    [3.05667963e+03, 0.00000000e+00, 1.00993159e+03, 0],
    [0.00000000e+00, 2.93188868e+03, 1.06438845e+03, 0],
    [0.00000000e+00, 0.00000000e+00, 1.00000000e+00, 0]
])
calib_matrix_P3 = calib_matrix_P2.copy()  # Use a matriz correta se tiver

cam1 = calib_matrix_P2[:, :3]
cam2 = calib_matrix_P3[:, :3]

# Parâmetros de rotação e translação
R = np.identity(3)
T = np.array([0.1, 0., 0.])

# Gere a matriz Q (reprojeção 3D)
h, w = imgL_gray.shape
Q = np.zeros((4, 4))
cv.stereoRectify(
    cameraMatrix1=cam1, cameraMatrix2=cam2,
    distCoeffs1=None, distCoeffs2=None,
    imageSize=(w, h),
    R=R, T=T,
    R1=None, R2=None, P1=None, P2=None, Q=Q,
    flags=0, alpha=0
)

print("Matriz Q (reprojeção 3D):")
print(Q)

In [ ]:
plt.imshow(disparity, cmap='gray')
plt.colorbar()
plt.title('Mapa de Disparidade')
plt.show()

In [ ]:
# Gere a nuvem de pontos 3D
points_3D = cv.reprojectImageTo3D(disparity, Q)
colors = cv.cvtColor(imgL, cv.COLOR_BGR2RGB)
mask = (disparity > 0) & np.isfinite(disparity)
out_points = points_3D[mask]
out_colors = colors[mask]

# Função para salvar em PLY
def write_ply(filename, verts, colors):
    verts = verts.reshape(-1, 3)
    colors = colors.reshape(-1, 3)
    verts = np.hstack([verts, colors])
    ply_header = '''ply
format ascii 1.0
element vertex %(vert_num)d
property float x
property float y
property float z
property uchar red
property uchar green
property uchar blue
end_header
'''
    with open(filename, 'w') as f:
        f.write(ply_header % dict(vert_num=len(verts)))
        np.savetxt(f, verts, '%f %f %f %d %d %d')

write_ply('out.ply', out_points, out_colors)
print('PLY file salvo!')

In [ ]:
idx = np.random.choice(len(out_points), size=min(10000, len(out_points)), replace=False)
pts = np.asarray(out_points)[idx]
zs = pts[:, 2]
cols = plt.cm.jet((zs - zs.min()) / (zs.ptp() + 1e-8))  # cor por profundidade

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=cols, s=0.5)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.imshow(disparity, cmap='gray')
plt.colorbar()
plt.title('Mapa de Disparidade')
plt.show()

In [ ]:
print("Shape dos pontos:", out_points.shape)
print("Coordenadas mínimas (X, Y, Z):", np.min(out_points, axis=0))
print("Coordenadas máximas (X, Y, Z):", np.max(out_points, axis=0))
print("Média (X, Y, Z):", np.mean(out_points, axis=0))
print("Desvio padrão (X, Y, Z):", np.std(out_points, axis=0))

In [ ]:
plt.hist(out_points[:,2], bins=100, color='blue', alpha=0.7)
plt.xlabel('Profundidade (Z)')
plt.ylabel('Frequência')
plt.title('Histograma da profundidade dos pontos')
plt.show()

In [ ]:
# Reflete X se necessário (opcional, depende do seu reprojectImageTo3D)
reflected_points = out_points.copy()
# reflected_points[:, 0] *= -1

# Projeta os pontos 3D de volta para 2D usando a matriz da câmera
projected_points, _ = cv.projectPoints(
    reflected_points, np.zeros(3), np.zeros(3), 
    calib_matrix_P2[:, :3], np.zeros(5)
)
projected_points = projected_points.reshape(-1, 2)

# Crie uma imagem em branco do mesmo tamanho da original
blank_img = np.zeros_like(imgL)

# Plote cada ponto na imagem
for i, pt in enumerate(projected_points):
    pt_x = int(round(pt[0]))
    pt_y = int(round(pt[1]))
    if 0 <= pt_x < blank_img.shape[1] and 0 <= pt_y < blank_img.shape[0]:
        col = tuple(int(c) for c in out_colors[i])
        cv.circle(blank_img, (pt_x, pt_y), 1, col, -1)

plt.figure(figsize=(10, 8))
plt.imshow(cv.cvtColor(blank_img, cv.COLOR_BGR2RGB))
plt.title("Pontos reprojetados na imagem")
plt.axis('off')
plt.show()

In [ ]:
idx = np.random.choice(len(out_points), size=min(20000, len(out_points)), replace=False)
pts = np.asarray(out_points)[idx]
cols = np.asarray(out_colors)[idx] / 255.0

fig = go.Figure(data=[go.Scatter3d(
    x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
    mode='markers',
    marker=dict(
        size=1.5,
        color=cols,  # RGB colors
        opacity=0.8
    )
)])

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        xaxis=dict(showbackground=True, backgroundcolor="rgb(230, 230,230)", gridcolor='gray'),
        yaxis=dict(showbackground=True, backgroundcolor="rgb(230, 230,230)", gridcolor='gray'),
        zaxis=dict(showbackground=True, backgroundcolor="rgb(230, 230,230)", gridcolor='gray'),
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    title="Nuvem de pontos 3D interativa"
)

fig.show()